# Explorer 06 - Networks and communities

**Purpose.** Build the collaboration, credit, and artist-label graphs; compute centrality; detect communities with Leiden; compare them with genre tags.

**Feeds main report section:** 5.5 RQ4 - Communities

**Modules used:** `vinyl.networks`

**Tables read:** `release_artists`, `release_credits`, `release_labels`, `release_genres`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-06-networks")
from vinyl import networks

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_artists = ingest.read_table(spark, snapshot, "release_artists")
release_credits = ingest.read_table(spark, snapshot, "release_credits")
release_labels = ingest.read_table(spark, snapshot, "release_labels")
release_genres = ingest.read_table(spark, snapshot, "release_genres")

## 1. Edge lists (Spark)

Check the edge count before collecting. Raise `min_weight` until the graph fits comfortably on the driver.

In [ ]:
collab = networks.collaboration_edges(release_artists, min_weight=2, max_artists_per_release=20).cache()
credit = networks.credit_edges(release_artists, release_credits, roles=config.PRODUCTION_ROLES, min_weight=2).cache()
artist_label = networks.artist_label_edges(release_artists, release_labels, min_weight=2).cache()
for name, edges in [("collaboration", collab), ("credit", credit), ("artist-label", artist_label)]:
    print(name, edges.count())

In [ ]:
collab.groupBy("weight").count().orderBy("weight").show(20)   # weight distribution -> choose min_weight

## 2. Centrality (driver)

In [ ]:
graph = networks.to_igraph(collab.toPandas())
print(graph.summary())
centrality = networks.centrality_table(graph, betweenness=False)
centrality.sort_values("pagerank", ascending=False).head(20)

## 3. Leiden communities

In [ ]:
communities = networks.leiden_communities(graph, resolution=1.0, seed=42)
print("modularity:", communities.attrs["modularity"])
communities.groupby("community").size().sort_values(ascending=False).head(20)

## 4. Do communities match genre tags?

In [ ]:
# Each artist's dominant genre = the genre on most of their releases.
from pyspark.sql import Window

counts = (release_artists.join(release_genres.select("release_id", "genre"), "release_id")
          .groupBy("artist_id", "genre").count())
top = counts.withColumn("r", F.row_number().over(Window.partitionBy("artist_id").orderBy(F.desc("count"), "genre")))
node_tags = top.where("r = 1").select(F.col("artist_id").alias("node"), "genre").toPandas()

networks.community_tag_agreement(communities, node_tags, tag_col="genre")

In [ ]:
# TODO: name the largest communities (top artists by pagerank + top styles inside each).
# TODO: repeat for the credit graph and the artist-label graph (prefix ids: "a123" / "l123").
# TODO: resolution sweep (0.5, 1, 2) - how stable are the communities?
# TODO: pull artist names from the artists table for readable tables.

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] `min_weight` and `max_artists_per_release` for each graph, and resulting node/edge counts.
- [ ] Leiden resolution, seed, and modularity.
- [ ] Agreement scores (NMI, ARI, purity) against genre and against style.
- [ ] Names for the communities shown in the report.